# Using the recommender: `train`, `predict`, `recommend`
Three functions are all the rest of the team needs. IDs are the **original** user/item ids; no integer indexing.

| call | returns |
|---|---|
| `train(ratings_df, weights=None, params=None)` | a trained model |
| `model.predict(user_id, item_id)` | predicted rating in [1, 5] |
| `model.recommend(user_id, n=10, exclude_seen=True)` | ranked list of item ids, best first |

Run this notebook from the folder that **contains** `recommender/` (the project root folder containing `recommender/`).

In [1]:
import numpy as np, pandas as pd
from recommender import train, RecommenderModel, data

# Real sample if you made it with make_sample.py, otherwise a small synthetic stand-in.
import os
path = "recommender/data/ratings_sample.csv"
if os.path.exists(path):
    ratings = pd.read_csv(path, dtype={"user_id": str, "item_id": str})
else:
    ratings = data.k_core(data.make_synthetic_ratings(n_users=6000, n_items=2000, density_target=0.004), 5).reset_index(drop=True)
ratings["review_id"] = np.arange(len(ratings))          # shared key used by the weights file
print(ratings.shape); ratings.head()

(36630, 5)


,user_id,item_id,rating,timestamp,review_id
0,U02417,I01017,5.0,1043812296,0
1,U04657,I00085,3.0,1164069922,1
2,U00494,I00473,5.0,1002156081,2
3,U05064,I01702,4.0,1036484209,3
4,U05137,I01612,2.0,1102773852,4


## 1. Train (FunkSVD is the default; use `params={'model': 'als'}` for ALS)

In [2]:
model = train(ratings, params={"model": "funksvd", "k": 10, "reg": 0.2, "epochs": 20})
model

RecommenderModel(funksvd, users=3000, items=1774, params={'model': 'funksvd', 'k': 10, 'reg': 0.2, 'lr': 0.005, 'epochs': 20, 'patience': 3, 'seed': 42})

## 2. Predict a rating

In [3]:
u, i = ratings.user_id.iloc[0], ratings.item_id.iloc[0]
print("actual rating   :", ratings.rating.iloc[0])
print("predicted rating:", round(model.predict(u, i), 3))
print("many at once    :", model.predict(ratings.user_id.iloc[:5].tolist(), ratings.item_id.iloc[:5].tolist()).round(2))
print("unknown user    :", round(model.predict('someone-new', i), 3), "(falls back to global mean + item bias)")

actual rating   : 5.0
predicted rating: 3.475
many at once    : [3.48 3.52 4.42 3.02 3.44]
unknown user    : 3.604 (falls back to global mean + item bias)


## 3. Recommend

In [4]:
print("top 10 for", u, ":", model.recommend(u, n=10))
print("their own rated items are excluded:", not set(model.recommend(u, 10)) & set(ratings[ratings.user_id == u].item_id))
print("unknown user gets the most-rated items:", model.recommend('someone-new', n=3))

top 10 for U02417 : ['I01173', 'I00765', 'I01694', 'I01150', 'I00621', 'I01412', 'I00162', 'I01252', 'I00867', 'I01875']
their own rated items are excluded: True
unknown user gets the most-rated items: ['I00510', 'I00778', 'I00247']


## 4. With weights (fake-review filtering)
`weights` is either an array aligned to the rows of `ratings`, or a DataFrame with `review_id, w` (the team's weights file).
`w` is in [0, 1], high = trusted. `w = 0` behaves exactly like deleting the rating.

In [5]:
# Example: pretend the detector flagged 5% of ratings as fake (hard filter) -- replace with the real weights.csv
rng = np.random.default_rng(0)
p_fake = rng.random(len(ratings)) ** 6                                  # stand-in detector scores
from recommender.weights import make_weights
w_hard = make_weights(p_fake, mode="hard", threshold=0.5)
w_soft = make_weights(p_fake, mode="soft", gamma=1.0)
print("removed by hard filter:", int((w_hard == 0).sum()), "| mean soft weight:", w_soft.mean().round(3))

model_als = train(ratings, weights=w_soft, params={"model": "als", "k": 20, "reg": 1.0})
print(model_als.recommend(u, n=5))

# or pass the team's file directly:
# weights_df = pd.read_csv("weights.csv")          # columns: review_id, w
# model_als = train(ratings, weights=weights_df, params={"model": "als"})

removed by hard filter: 3881 | mean soft weight: 0.858


['I00830', 'I00621', 'I00233', 'I01388', 'I01173']


## 5. Save and load

In [6]:
model.save("model.pkl")
same = RecommenderModel.load("model.pkl")
print(np.array_equal(model.predict(u, i), same.predict(u, i)))
os.remove("model.pkl")

True
